# H5 Data Loading and Organization

This notebook loads raw H5 files from multiple subjects and trials, organizing them into a structured DataFrame for downstream analysis.

## Overview
- **Input**: Multiple `.h5` files with naming pattern `{subject_id}_{trial_id}.h5`
- **Output**: Pandas DataFrame with structured metadata and data references
- **Features**: Automatic file discovery, error handling, sanity checks, and visualizations

## File Naming Convention
- Pattern: `XX_YY.h5` where:
  - `XX` = subject ID (e.g., 00, 01, 02, ...)
  - `YY` = trial ID (e.g., 00, 01, 02, ..., 05)
- Example: `00_00.h5`, `00_01.h5`, ..., `01_00.h5`, etc.

## 1. Configuration and Imports

In [ ]:
# Standard library imports
import re
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import warnings

# Third-party imports
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

In [ ]:
# ============================================
# CONFIGURATION: Set your data path here
# ============================================

# Base path containing the H5 files
# Replace this with your actual data directory
BASE_PATH = "/Users/danieleuras/Library/CloudStorage/OneDrive-PolitecnicodiMilano/File di Francesco Iacomi - h5/data"

# Convert to Path object for cross-platform compatibility
data_dir = Path(BASE_PATH)

# Optional: Path to electrode location file (for channel naming)
# This path is relative to the project root
project_root = Path.cwd().parent
eloc_path = project_root / "scripts" / "data_processing" / "Preprocessing" / "ebneuro.eloc"

print(f"Data directory: {data_dir}")
print(f"Data directory exists: {data_dir.exists()}")
print(f"Electrode location file: {eloc_path}")
print(f"Electrode file exists: {eloc_path.exists()}")

## 2. Helper Functions

These functions handle file discovery, parsing, and data loading.

In [ ]:
def discover_h5_files(base_path: Path, pattern: str = r"^(\d{2})_(\d{2})\.h5$") -> List[Tuple[Path, str, str]]:
    """
    Discover all H5 files matching the subject-trial naming pattern.
    
    Args:
        base_path: Directory containing H5 files
        pattern: Regular expression pattern for matching filenames
        
    Returns:
        List of tuples: (file_path, subject_id, trial_id)
        
    Example:
        >>> files = discover_h5_files(Path("/data"))
        >>> # Returns: [(Path("/data/00_00.h5"), "00", "00"), ...]
    """
    if not base_path.exists():
        raise FileNotFoundError(f"Data directory not found: {base_path}")
    
    # Compile regex pattern
    regex = re.compile(pattern)
    
    discovered_files = []
    
    # Search for H5 files
    for h5_file in sorted(base_path.glob("*.h5")):
        match = regex.match(h5_file.name)
        if match:
            subject_id = match.group(1)
            trial_id = match.group(2)
            discovered_files.append((h5_file, subject_id, trial_id))
    
    if not discovered_files:
        print(f"WARNING: No H5 files found matching pattern '{pattern}' in {base_path}")
    
    return discovered_files


def load_h5_file_info(file_path: Path) -> Dict:
    """
    Load metadata and basic information from an H5 file.
    
    Args:
        file_path: Path to H5 file
        
    Returns:
        Dictionary containing file metadata
        
    Raises:
        Exception: If file cannot be read or has unexpected structure
    """
    try:
        with h5py.File(file_path, "r") as f:
            # Get available keys in the H5 file
            available_keys = list(f.keys())
            
            # Initialize metadata dictionary
            metadata = {
                "file_path": str(file_path),
                "file_name": file_path.name,
                "available_keys": available_keys,
                "file_size_mb": file_path.stat().st_size / (1024 * 1024)
            }
            
            # Try to load standard keys (adapt based on actual H5 structure)
            if "data" in f:
                data_shape = f["data"].shape
                metadata["data_shape"] = data_shape
                metadata["n_epochs"] = data_shape[0]
                metadata["n_channels"] = data_shape[1]
                metadata["n_samples"] = data_shape[2]
            
            if "labels" in f:
                labels = f["labels"][:]
                metadata["n_labels"] = len(labels)
                # Decode first label as sample
                if len(labels) > 0:
                    first_label = labels[0]
                    if isinstance(first_label, (bytes, bytearray)):
                        first_label = first_label.decode('utf-8', errors='ignore')
                    metadata["sample_label"] = str(first_label)
            
            if "subject" in f:
                subject = f["subject"][()]
                if isinstance(subject, (bytes, bytearray)):
                    subject = subject.decode('utf-8', errors='ignore')
                metadata["subject_from_file"] = str(subject)
            
            return metadata
            
    except Exception as e:
        return {
            "file_path": str(file_path),
            "file_name": file_path.name,
            "error": str(e),
            "status": "failed"
        }


def load_h5_data(file_path: Path, load_full_data: bool = False) -> Dict:
    """
    Load data and labels from an H5 file.
    
    Args:
        file_path: Path to H5 file
        load_full_data: If True, loads entire data array into memory.
                       If False, returns reference to data in file.
        
    Returns:
        Dictionary containing data, labels, and metadata
    """
    result = {"file_path": str(file_path), "status": "success"}
    
    try:
        with h5py.File(file_path, "r") as f:
            # Load or reference data
            if "data" in f:
                if load_full_data:
                    result["data"] = f["data"][:]
                else:
                    # Store shape info only
                    result["data_shape"] = f["data"].shape
            
            # Load labels
            if "labels" in f:
                labels = f["labels"][:]
                # Decode labels if they are bytes
                decoded_labels = []
                for label in labels:
                    if isinstance(label, (bytes, bytearray)):
                        label = label.decode('utf-8', errors='ignore')
                    decoded_labels.append(str(label).strip())
                result["labels"] = decoded_labels
            
            # Load subject info
            if "subject" in f:
                subject = f["subject"][()]
                if isinstance(subject, (bytes, bytearray)):
                    subject = subject.decode('utf-8', errors='ignore')
                result["subject"] = str(subject)
                
    except Exception as e:
        result["status"] = "failed"
        result["error"] = str(e)
    
    return result


def load_channel_names(eloc_path: Path) -> List[str]:
    """
    Load channel names from .eloc file.
    
    Args:
        eloc_path: Path to .eloc file
        
    Returns:
        List of channel names
    """
    if not eloc_path.exists():
        print(f"WARNING: Electrode file not found: {eloc_path}")
        return []
    
    try:
        df = pd.read_csv(eloc_path, sep=r"\s+", header=None, engine="python")
        names = df.iloc[:, -1].astype(str).tolist()
        return names
    except Exception as e:
        print(f"ERROR loading electrode file: {e}")
        return []

## 3. Build Dataset Structure

Create a structured DataFrame containing all files and their metadata.

In [ ]:
def build_dataset_dataframe(base_path: Path, verbose: bool = True) -> pd.DataFrame:
    """
    Build a comprehensive DataFrame from all H5 files in the directory.
    
    Args:
        base_path: Directory containing H5 files
        verbose: Print progress information
        
    Returns:
        DataFrame with columns: subject_id, trial_id, file_path, data_shape, labels, etc.
    """
    # Discover all H5 files
    if verbose:
        print("Discovering H5 files...")
    
    discovered_files = discover_h5_files(base_path)
    
    if not discovered_files:
        print("No files found. Returning empty DataFrame.")
        return pd.DataFrame()
    
    if verbose:
        print(f"Found {len(discovered_files)} H5 files.")
        print("Loading file metadata...\n")
    
    # Load metadata for each file
    rows = []
    failed_files = []
    
    for file_path, subject_id, trial_id in discovered_files:
        if verbose:
            print(f"Processing: {file_path.name} (Subject: {subject_id}, Trial: {trial_id})")
        
        # Load file metadata
        file_info = load_h5_file_info(file_path)
        
        # Check for errors
        if "error" in file_info:
            failed_files.append((file_path.name, file_info["error"]))
            if verbose:
                print(f"  ⚠️  ERROR: {file_info['error']}\n")
            continue
        
        # Create row with metadata
        row = {
            "subject_id": subject_id,
            "trial_id": trial_id,
            "file_path": str(file_path),
            "file_name": file_path.name,
            "file_size_mb": file_info.get("file_size_mb", 0),
            "n_epochs": file_info.get("n_epochs", 0),
            "n_channels": file_info.get("n_channels", 0),
            "n_samples": file_info.get("n_samples", 0),
            "data_shape": str(file_info.get("data_shape", "N/A")),
            "sample_label": file_info.get("sample_label", "N/A"),
            "available_keys": ",".join(file_info.get("available_keys", []))
        }
        
        rows.append(row)
        
        if verbose:
            print(f"  ✓ Shape: {file_info.get('data_shape', 'N/A')}")
            print(f"  ✓ Sample label: {file_info.get('sample_label', 'N/A')}\n")
    
    # Create DataFrame
    df = pd.DataFrame(rows)
    
    # Sort by subject and trial
    if not df.empty:
        df = df.sort_values(["subject_id", "trial_id"]).reset_index(drop=True)
    
    # Print summary
    if verbose:
        print("=" * 60)
        print("SUMMARY")
        print("=" * 60)
        print(f"Total files processed: {len(discovered_files)}")
        print(f"Successfully loaded: {len(rows)}")
        print(f"Failed: {len(failed_files)}")
        
        if failed_files:
            print("\nFailed files:")
            for fname, error in failed_files:
                print(f"  - {fname}: {error}")
    
    return df

## 4. Load the Dataset

Execute the data loading process.

In [ ]:
# Build the dataset DataFrame
dataset_df = build_dataset_dataframe(data_dir, verbose=True)

In [ ]:
# Display the first few rows
print("\nDataset Preview:")
print("=" * 80)
dataset_df.head(10)

## 5. Sanity Checks and Statistics

Validate the loaded data and compute useful statistics.

In [ ]:
def perform_sanity_checks(df: pd.DataFrame) -> Dict:
    """
    Perform comprehensive sanity checks on the loaded dataset.
    
    Args:
        df: Dataset DataFrame
        
    Returns:
        Dictionary containing check results and statistics
    """
    if df.empty:
        return {"status": "empty", "message": "Dataset is empty"}
    
    checks = {}
    
    # 1. Number of subjects
    n_subjects = df["subject_id"].nunique()
    checks["n_subjects"] = n_subjects
    
    # 2. Number of trials per subject
    trials_per_subject = df.groupby("subject_id")["trial_id"].nunique()
    checks["trials_per_subject"] = trials_per_subject.to_dict()
    checks["min_trials"] = trials_per_subject.min()
    checks["max_trials"] = trials_per_subject.max()
    checks["mean_trials"] = trials_per_subject.mean()
    
    # 3. Check for consistent data shapes
    if "n_channels" in df.columns:
        unique_n_channels = df["n_channels"].unique()
        checks["unique_n_channels"] = unique_n_channels.tolist()
        checks["consistent_channels"] = len(unique_n_channels) == 1
    
    if "n_samples" in df.columns:
        unique_n_samples = df["n_samples"].unique()
        checks["unique_n_samples"] = unique_n_samples.tolist()
        checks["consistent_samples"] = len(unique_n_samples) == 1
    
    # 4. Check for missing trials
    missing_trials = []
    for subject_id in df["subject_id"].unique():
        subject_trials = df[df["subject_id"] == subject_id]["trial_id"].astype(int).values
        expected_trials = set(range(subject_trials.min(), subject_trials.max() + 1))
        actual_trials = set(subject_trials)
        missing = expected_trials - actual_trials
        if missing:
            missing_trials.append((subject_id, sorted(list(missing))))
    
    checks["missing_trials"] = missing_trials
    checks["has_missing_trials"] = len(missing_trials) > 0
    
    # 5. Total dataset size
    if "file_size_mb" in df.columns:
        checks["total_size_mb"] = df["file_size_mb"].sum()
        checks["avg_file_size_mb"] = df["file_size_mb"].mean()
    
    # 6. Total epochs across all files
    if "n_epochs" in df.columns:
        checks["total_epochs"] = df["n_epochs"].sum()
    
    return checks

In [ ]:
# Run sanity checks
checks = perform_sanity_checks(dataset_df)

# Display results
print("\n" + "=" * 60)
print("SANITY CHECKS & STATISTICS")
print("=" * 60)

if dataset_df.empty:
    print("⚠️  Dataset is empty. Check your BASE_PATH configuration.")
else:
    print(f"\n📊 Dataset Statistics:")
    print(f"  • Number of subjects: {checks.get('n_subjects', 'N/A')}")
    print(f"  • Trials per subject (min/max/mean): {checks.get('min_trials', 'N/A')} / {checks.get('max_trials', 'N/A')} / {checks.get('mean_trials', 'N/A'):.1f}")
    print(f"  • Total epochs across all files: {checks.get('total_epochs', 'N/A')}")
    print(f"  • Total dataset size: {checks.get('total_size_mb', 0):.2f} MB")
    
    print(f"\n🔍 Data Consistency:")
    print(f"  • Consistent number of channels: {'✓ Yes' if checks.get('consistent_channels') else '✗ No'}")
    if not checks.get('consistent_channels'):
        print(f"    Unique channel counts: {checks.get('unique_n_channels', [])}")
    else:
        print(f"    Number of channels: {checks.get('unique_n_channels', ['N/A'])[0]}")
    
    print(f"  • Consistent number of samples: {'✓ Yes' if checks.get('consistent_samples') else '✗ No'}")
    if not checks.get('consistent_samples'):
        print(f"    Unique sample counts: {checks.get('unique_n_samples', [])}")
    else:
        print(f"    Number of samples per epoch: {checks.get('unique_n_samples', ['N/A'])[0]}")
    
    print(f"\n⚠️  Missing Trials:")
    if checks.get('has_missing_trials'):
        print(f"  Found missing trials for {len(checks['missing_trials'])} subject(s):")
        for subject_id, missing in checks['missing_trials']:
            print(f"    • Subject {subject_id}: missing trials {missing}")
    else:
        print(f"  ✓ No missing trials detected")
    
    print(f"\n📋 Trials per subject:")
    for subject_id, n_trials in sorted(checks.get('trials_per_subject', {}).items()):
        print(f"  • Subject {subject_id}: {n_trials} trials")

## 6. Visualizations

Create visualizations to better understand the dataset structure.

In [ ]:
# Skip visualizations if dataset is empty
if not dataset_df.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Dataset Overview', fontsize=16, fontweight='bold')
    
    # 1. Number of trials per subject
    trials_count = dataset_df.groupby('subject_id').size()
    ax1 = axes[0, 0]
    trials_count.plot(kind='bar', ax=ax1, color='steelblue')
    ax1.set_title('Number of Trials per Subject')
    ax1.set_xlabel('Subject ID')
    ax1.set_ylabel('Number of Trials')
    ax1.grid(axis='y', alpha=0.3)
    
    # 2. File size distribution
    if 'file_size_mb' in dataset_df.columns:
        ax2 = axes[0, 1]
        dataset_df['file_size_mb'].hist(bins=20, ax=ax2, color='coral', edgecolor='black')
        ax2.set_title('File Size Distribution')
        ax2.set_xlabel('File Size (MB)')
        ax2.set_ylabel('Frequency')
        ax2.grid(axis='y', alpha=0.3)
    
    # 3. Number of epochs per file
    if 'n_epochs' in dataset_df.columns:
        ax3 = axes[1, 0]
        dataset_df['n_epochs'].hist(bins=20, ax=ax3, color='mediumseagreen', edgecolor='black')
        ax3.set_title('Number of Epochs per File')
        ax3.set_xlabel('Number of Epochs')
        ax3.set_ylabel('Frequency')
        ax3.grid(axis='y', alpha=0.3)
    
    # 4. Heatmap of trials per subject
    ax4 = axes[1, 1]
    pivot_data = dataset_df.pivot_table(
        index='subject_id', 
        columns='trial_id', 
        values='file_name', 
        aggfunc='count',
        fill_value=0
    )
    sns.heatmap(pivot_data, annot=True, fmt='d', cmap='YlGnBu', ax=ax4, cbar_kws={'label': 'File exists'})
    ax4.set_title('Trial Availability Heatmap')
    ax4.set_xlabel('Trial ID')
    ax4.set_ylabel('Subject ID')
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Skipping visualizations: Dataset is empty")

## 7. Example: Load and Visualize Data from One File

Load actual EEG data from a single file and create a sample visualization.

In [ ]:
# Load channel names if available
channel_names = load_channel_names(eloc_path)
if channel_names:
    print(f"Loaded {len(channel_names)} channel names from electrode file")
    print(f"Sample channels: {channel_names[:10]}...")
else:
    print("No channel names loaded. Will use generic names (EEG1, EEG2, ...)")

In [ ]:
# Select first file for example visualization
if not dataset_df.empty:
    # Get the first file
    first_file = Path(dataset_df.iloc[0]['file_path'])
    subject_id = dataset_df.iloc[0]['subject_id']
    trial_id = dataset_df.iloc[0]['trial_id']
    
    print(f"Loading example data from:")
    print(f"  File: {first_file.name}")
    print(f"  Subject: {subject_id}, Trial: {trial_id}")
    
    # Load the data
    data_dict = load_h5_data(first_file, load_full_data=True)
    
    if data_dict['status'] == 'success' and 'data' in data_dict:
        data = data_dict['data']
        labels = data_dict.get('labels', [])
        
        print(f"\nData loaded successfully!")
        print(f"  Shape: {data.shape}")
        print(f"  Data type: {data.dtype}")
        print(f"  Number of labels: {len(labels)}")
        if labels:
            print(f"  Sample labels: {labels[:5]}")
        
        # Plot first epoch, first few channels
        n_channels_to_plot = min(5, data.shape[1])
        
        fig, ax = plt.subplots(figsize=(14, 8))
        
        # Assuming sampling rate of 256 Hz (adjust if different)
        fs = 256
        time = np.arange(data.shape[2]) / fs
        
        # Plot first epoch
        for ch_idx in range(n_channels_to_plot):
            # Add offset for visualization
            offset = ch_idx * 50
            
            # Get channel name
            if channel_names and ch_idx < len(channel_names):
                ch_name = channel_names[ch_idx]
            else:
                ch_name = f"Channel {ch_idx + 1}"
            
            ax.plot(time, data[0, ch_idx, :] + offset, label=ch_name, alpha=0.8)
        
        ax.set_xlabel('Time (s)', fontsize=12)
        ax.set_ylabel('Amplitude (µV) + offset', fontsize=12)
        ax.set_title(f'Example EEG Data - First Epoch\nSubject: {subject_id}, Trial: {trial_id}', 
                    fontsize=14, fontweight='bold')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
        
        # Add label if available
        if labels:
            ax.text(0.02, 0.98, f"Label: {labels[0]}", 
                   transform=ax.transAxes, 
                   fontsize=11, 
                   verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        plt.tight_layout()
        plt.show()
        
    else:
        print(f"❌ Failed to load data: {data_dict.get('error', 'Unknown error')}")
else:
    print("⚠️  No files available for visualization")

## 8. Save Dataset Metadata

Optionally save the dataset metadata to a CSV file for future use.

In [ ]:
# Define output path
output_dir = project_root / "data" / "interim"
output_path = output_dir / "h5_dataset_metadata.csv"

# Create directory if it doesn't exist
output_dir.mkdir(parents=True, exist_ok=True)

# Save DataFrame
if not dataset_df.empty:
    dataset_df.to_csv(output_path, index=False)
    print(f"✓ Dataset metadata saved to: {output_path}")
    print(f"  Rows: {len(dataset_df)}")
    print(f"  Columns: {list(dataset_df.columns)}")
else:
    print("⚠️  Dataset is empty. No file saved.")

## 9. Advanced Usage: Build Dictionary Structure

Alternative data structure: nested dictionary organized by subject and trial.

In [ ]:
def build_nested_dict_structure(df: pd.DataFrame) -> Dict:
    """
    Build a nested dictionary structure: data[subject_id][trial_id] = {...}
    
    Args:
        df: Dataset DataFrame
        
    Returns:
        Nested dictionary with subject and trial organization
    """
    data_dict = {}
    
    for _, row in df.iterrows():
        subject_id = row['subject_id']
        trial_id = row['trial_id']
        
        # Initialize subject if not exists
        if subject_id not in data_dict:
            data_dict[subject_id] = {}
        
        # Add trial data
        data_dict[subject_id][trial_id] = {
            'file_path': row['file_path'],
            'file_name': row['file_name'],
            'n_epochs': row.get('n_epochs', 0),
            'n_channels': row.get('n_channels', 0),
            'n_samples': row.get('n_samples', 0),
            'data_shape': row.get('data_shape', 'N/A'),
        }
    
    return data_dict

# Build nested structure
if not dataset_df.empty:
    nested_data = build_nested_dict_structure(dataset_df)
    
    print("Nested dictionary structure created:")
    print(f"  Number of subjects: {len(nested_data)}")
    print(f"\nExample access:")
    first_subject = list(nested_data.keys())[0]
    first_trial = list(nested_data[first_subject].keys())[0]
    print(f"  nested_data['{first_subject}']['{first_trial}'] = ")
    print(f"    {nested_data[first_subject][first_trial]}")
else:
    print("⚠️  Cannot build nested structure: Dataset is empty")

## Summary

This notebook provides a comprehensive framework for loading and organizing H5 data files:

1. **Configuration**: Single `BASE_PATH` variable for easy customization
2. **Discovery**: Automatic detection of H5 files matching subject-trial pattern
3. **Loading**: Robust loading with error handling and metadata extraction
4. **Structure**: Both DataFrame and nested dictionary structures available
5. **Validation**: Comprehensive sanity checks for data consistency
6. **Visualization**: Multiple plots to understand dataset structure
7. **Integration**: Compatible with existing repository conventions

### Next Steps

- Adjust `BASE_PATH` to point to your actual data directory
- Run all cells to load and validate your dataset
- Use `dataset_df` for further preprocessing and analysis
- Integrate with existing notebooks (e.g., `dataframe.ipynb`, `feature_extraction_tutorial.ipynb`)

### Notes

- The notebook handles missing files and inconsistent naming gracefully
- Channel names are loaded from `.eloc` file if available
- Data is not fully loaded into memory by default (only metadata)
- Use `load_h5_data(file_path, load_full_data=True)` to load actual data when needed